## RAG - Routing
In the [Query Translation](02_Query_Translation.ipynb) notebook we covered several techniques for translating user's query - techniques such as Multi-Query, RAG Fusion, Query-Decomposition, Step Back and HYDE were discussed. The goal such techniques is to take input user question and translate it in such a way as to improve retrieval. In this notebook, we'll discuss the next step in the RAG pipeline - _Routing_.

<center>
<img src="../images/rag_detailed_pipeline.png" width="800" height="480"/>
</center>

### What is Routing?
**Routing** is the next step, which is potentially _directing_ the translated user query to the right source. We could have several sources of data that the user would like to query from, sources such as vector-stores,  GraphDBs or an RDBMS. We simply route (or direct) the query to the right source based upon content of the question. There are a few different ways to do that.

One of the techniques is called **Logical Routing**. In this case we basically give our LLM knowledge of the various datasources that we have at our disposal and we let the LLM _reason_ about which one to apply the question to.

<center>
<img src="../images/logical_routing.png"/>
</center>

Alternatively, we could use **Semantic Routing**, which is where we take take the user's question/query, we embed it, we also embed prompts and compare the similarity betweeen our question and embedded prompts and we choose a prompt based on the similarity.

<center>
<img src="../images/semantic_routing.png"/>
</center>

So the general idea is to route question to different prompts (or arbitrarily taking the question and sending it to the rioht source that can answer it).

In this notebook, we will cover both _Logical_ and _Semantic_ routing techniques. We'll be using the LangChain framework with Google Gemini 2.5 Flash LLM, but you can always replace it with an LLM of your choice below. 

In [2]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from IPython.display import display, Markdown

from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

In [3]:
# load API keys from .env files
load_dotenv(override=True)
console = Console()

In [4]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)

### Logical Routing

A business scenario that could explain the need for Logical routing would be something like the following: Let's say _WeCoverAnything_ (WCA) a [fictitious] Insurance company tha sells Life Insurance, Health Insurance and Car Insurance policies. Bob is a client who has bought Health _and_ Car insurnace policies from WCA. WCA has deployed a customer facing chatbot (powered by an LLM of course!) that can answer common questions from end-customers such as Bob (or redirect their query to human Helpdesk agents in case it cannot answer the question). Bob could ask a question like "Is my policy covering both own damage and third-party liability? Can you explain what each one means?" or something like "I need to file a claim for a recent hospitalization. What documents do I need, and how do I submit them?". Clearly the first relates to Car Insurance and the second to Health Insurance. The chabot must be intelligent enough to direct the former question to "Car Insurance Policy" datasources and the latter to "Health Insurance Policy" data sources.

Let's walk through a Logical Routing use-case. Suppose that we have 3 knowledge sources focused related to Python, JavaScript and Go programming respectively. So, we'd like to re-direct all Python queries to the Python datastore, JavaScript to the JS data store and so on. 

<center>
<img src="images/logical_routing.png"/>
</center>

We bind LLM output to a structure (a Pydantic datamodel), so that it returns one of a fixed set of outputs, which we can then use to redirect to specific datasource. We use the `llm_with_structured_output(...)` call to bind LLM's output, similar to what we did in the[Classification Example](../04_classificaltion.py)

In [18]:
from typing import Literal
from pydantic import BaseModel, Field


class UserQueryRouter(BaseModel):
    """Route a user query regarding insurance policy to car or health policy sources"""

    datasource: Literal["car_policy_docs", "health_policy_docs", "unknown"] = Field(
        ...,
        description="Given a user question regarding his/her car or health insurance policy choose which datasource would be most relevant for answering their policy related question. If the question is not related to an either car or health insurance, return 'unknown'",
    )


structured_llm_policy = llm.with_structured_output(UserQueryRouter)


# Data model: here we define various "routes" depending on programming language
# So Python related queries should go to "python_docs", JavaScript to "js_docs" etc.
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    # these could be paths of vector stores, or identifiers for APIs etc.
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )


structured_llm_code = llm.with_structured_output(RouteQuery)

In [21]:
# Prompt
system_prompt_policy = """You are an expert at routing a user question to the appropriate data source.

Based on the type of insurance policy (Car or Health) the question is referring to, route it to the relevant data source. If it is referring to neither, route it to 'unknown'."""

policy_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_policy),
        ("human", "{question}"),
    ]
)

# Define router
policy_router = policy_prompt | structured_llm_policy

In [22]:
# now let's try it out with a question specific to Car insurance
question_car_insurance = """
    Is my policy covering both own damage and third-party liability? 
    Can you explain what each one means?
"""

result = policy_router.invoke({"question": question_car_insurance})
print(result.datasource)

car_policy_docs


In [33]:
# now let's try it out with a question specific to Health Insurance
question_health_insurance = """
    I need to file a claim for a recent hospitalization. What documents do I need, and how do I submit them?
"""

result = policy_router.invoke({"question": question_health_insurance})
print(result.datasource)

health_policy_docs


In [34]:
# what if I ask an unrelated (to car or healh insurance) question?
question_random = """
    My laptop is freezing randomly. I just upgraded by RAM. Could it be a RAM issue?
"""

result = policy_router.invoke({"question": question_random})
print(result.datasource)

unknown


In [35]:
def choose_policy_route(result):
    if "car_policy_docs" in result.datasource.lower():
        ### Logic here
        return "chain for handling Car Insurance related query"
    elif "health_policy_docs" in result.datasource.lower():
        ### Logic here
        return "chain for handling Health Insurance related query"
    else:
        ### Logic here
        return (
            "Aplogies, I don't understand your question - routing to a human advisor."
        )


from langchain_core.runnables import RunnableLambda

full_policy_chain = policy_router | RunnableLambda(choose_policy_route)

In [36]:
full_policy_chain.invoke({"question": question_car_insurance})

'chain for handling Car Insurance related query'

In [37]:
full_policy_chain.invoke({"question": question_health_insurance})

'chain for handling Health Insurance related query'

In [40]:
# try out with any question
# full_policy_chain.invoke({"question": "Do you cove headlight repairs?"})
full_policy_chain.invoke({"question": "Do you cover cyclone damage of my home?"})

"Aplogies, I don't understand your question - routing to a human advisor."

In [38]:
full_policy_chain.invoke({"question": question_random})

"Aplogies, I don't understand your question - routing to a human advisor."

In [26]:
# Prompt
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router
router = prompt | structured_llm_code

In [27]:
# now let's try it out with various languages
# Python first
question_python = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question_python})
print(result.datasource)

python_docs


In [28]:
# How about this?
question_go = """Why doesn't the following code work:

import (
	"fmt"
)

func main() {
	m := make(map[string]int)
	vals := []int{1, 2, 3}

	for _, v := range vals {
		go func() {
			m["sum"] += v
		}()
	}

	fmt.Println("sum:", m["sum"])
}
"""

result = router.invoke({"question": question_go})
print(result.datasource)

golang_docs


In [29]:
# And this?
question_js = """Why doesn't the following code work:

let count = 0;

for (var i = 0; i < 5; i++) {
  setInterval(function () {
    count++;
    console.log("i:", i, "count:", count);
    if (count === 5) {
      clearInterval(this);
    }
  }, 1000);
}
"""

result = router.invoke({"question": question_js})
print(result.datasource)

js_docs


So as you can see, the LLM is able to _detect_ the programming language and _direct_ us to the correct data source to redirect our query to!

Now let us create a common function to route.

In [10]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        ### Logic here
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        ### Logic here
        return "chain for js_docs"
    else:
        ### Logic here
        return "golang_docs"


from langchain_core.runnables import RunnableLambda

full_chain = router | RunnableLambda(choose_route)

In [11]:
full_chain.invoke({"question": question_js})

'chain for js_docs'

In [12]:
full_chain.invoke({"question": question_python})

'chain for python_docs'

In [13]:
full_chain.invoke({"question": question_go})

'golang_docs'

### Semantic Routing

Semantic routing is a little bit straightforward as compared to Logical Routing. In this case we have a series of prompts (or more correctly, prompt templates) we want to choose depending on the input query - so say. we have a Physics related prompt-template and a Math related prompt-template and the user asks a question related to either Physics or Maths. 

First we embed both the templates as well as the user's question. Then we use a `cosine similarity` function to choose which subject the prompt is related to, and then fire the "closest match" prompt.

<center>
<img src="images/semantic_routing.png"/>
</center>

In [43]:
from langchain.utils.math import cosine_similarity
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_cohere import CohereEmbeddings

In [44]:
# Two prompt templates for different subjects
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

In [46]:
embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
)
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

In [47]:
# Route question to correct prompt
def prompt_router(input):
    # Embed question
    query_embedding = embeddings.embed_query(input["query"])
    # Compute similarity between embedded query and embedded prompt templates
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    print(f"Similarities -> {similarity}")
    most_similar = prompt_templates[similarity.argmax()]
    # Chosen prompt
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

In [48]:
prompt_router({"query": "What is Newton's second law of motion?"})  # a Physics question

Similarities -> [0.27495906 0.18490319]
Using PHYSICS


PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template="You are a very smart physics professor. You are great at answering questions about physics in a concise and easy to understand manner. When you don't know the answer to a question you admit that you don't know.\n\nHere is a question:\n{query}")

In [49]:
# now let's build our chain
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

In [50]:
chain.invoke("What is Newton's second law of motion?")  # Physics question

Similarities -> [0.27495906 0.18490319]
Using PHYSICS


"Newton's second law of motion states that the acceleration of an object is directly proportional to the net force acting on it and inversely proportional to its mass. The direction of the acceleration is in the direction of the net force.\n\nIn simpler terms, it's often expressed by the famous equation:\n\n**F = ma**\n\nWhere:\n*   **F** is the net force acting on the object (measured in Newtons).\n*   **m** is the mass of the object (measured in kilograms).\n*   **a** is the acceleration of the object (measured in meters per second squared).\n\nThis law tells us that if you push an object with more force, it will accelerate more. And if an object has more mass, you'll need to apply more force to get it to accelerate at the same rate."

In [51]:
chain.invoke("What is Pythagoras' theorem?")  # Math question

Similarities -> [0.20418755 0.2830592 ]
Using MATH


'Pythagoras\' theorem is a fundamental principle in geometry that describes a special relationship between the sides of a **right-angled triangle**.\n\nLet\'s break it down:\n\n### Component 1: The Context - What kind of triangle does it apply to?\n\n*   **Right-angled triangle:** This is a triangle that has one angle exactly equal to 90 degrees (a "right angle").\n\n### Component 2: The Sides - What are the parts of a right-angled triangle?\n\n*   **Hypotenuse:** This is the longest side of the right-angled triangle. It is always the side directly opposite the right angle.\n*   **Legs (or Cathetus):** These are the other two sides of the triangle that form the right angle.\n\n### Component 3: The Relationship - What does the theorem state?\n\nPythagoras\' theorem states that:\n\n**"In a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides (the legs)."**\n\n### Comp

In [52]:
# Math or Physics?
chain.invoke(
    "If a car travels at a constant speed of 60 miles per hour, how far will it travel in 3 hours?"
)

Similarities -> [0.11114148 0.09468728]
Using PHYSICS


"That's a straightforward one!\n\nIf a car travels at a constant speed of 60 miles per hour for 3 hours, it will travel:\n\nDistance = Speed × Time\nDistance = 60 mph × 3 h\n**Distance = 180 miles**"

In [53]:
chain.invoke(
    "What is the maximum volume of a sphere that can be contained within a cube of side length $L$, and how does the pressure exerted by an ideal gas inside that sphere relate to its temperature?"
)

Similarities -> [0.15424759 0.14749195]
Using PHYSICS


"Alright, let's break this down.\n\n1.  **Maximum Volume of the Sphere:**\n    For a sphere to be contained within a cube of side length $L$, its diameter must be no larger than $L$. To maximize the sphere's volume, its diameter must be exactly $L$.\n    So, the radius of the sphere, $R = L/2$.\n    The volume of a sphere is given by $V = \\frac{4}{3}\\pi R^3$.\n    Substituting $R = L/2$:\n    $V = \\frac{4}{3}\\pi \\left(\\frac{L}{2}\\right)^3 = \\frac{4}{3}\\pi \\frac{L^3}{8} = \\frac{\\pi L^3}{6}$.\n\n2.  **Pressure of an Ideal Gas and Temperature:**\n    For an ideal gas, the relationship between pressure ($P$) and absolute temperature ($T$) is given by the Ideal Gas Law: $PV = nRT$.\n    Here, $V$ is the volume of the gas (which is the sphere's volume), $n$ is the number of moles of gas, and $R$ is the ideal gas constant.\n    If the number of moles of gas ($n$) and the volume ($V$) are kept constant, then the pressure is directly proportional to the absolute temperature:\n    $P

In [28]:
from pydantic import BaseModel, Field


class InputQuestionsList(BaseModel):
    """Input schema for list of questions."""

    num_questions: int = Field(..., description="Number of user questions.")
    questions: list[str] = Field(..., description="List of user questions.")


structured_llm = llm.with_structured_output(InputQuestionsList)

In [31]:
prompt_template = """
From the following user input, extract the number questions asked, and extract the questions separately. User could ask multiple questions in a single input.

User Input: {user_input}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert at extracting questions from user input."),
        ("human", prompt_template),
    ]
)

question_chain = prompt | structured_llm

In [34]:
response = question_chain.invoke(
    {
        "user_input": "What is Newton's second law of motion? Also, what is Pythagoras' theorem?"
    }
)
print(f"I extracted {response.num_questions} questions.")
print("Here are the questions:")
for i in range(response.num_questions):
    print(f"{i+1} -> {response.questions[i]}")

I extracted 2 questions.
Here are the questions:
1 -> What is Newton's second law of motion?
2 -> what is Pythagoras' theorem?
